# Day 6. Loss / Cross Entropy

**오늘의 목표**
- Prediction, Target, Loss가 무엇인지 설명할 수 있다
- 분류 문제(= 다음 Token 고르기)에 MSE 대신 **Cross Entropy**를 쓰는 이유를 안다
- `-log(p)`가 "확신하고 맞히면 작고, 확신하고 틀리면 엄청 크다"는 걸 그래프로 본다
- Cross Entropy를 손으로 계산해서 `F.cross_entropy`와 똑같이 만들 수 있다
- GPT의 `(B, T, V)` 출력으로 Loss를 계산하는 코드를 읽을 수 있다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()]

## 1. Prediction / Target / Loss

```text
모델 예측 (Prediction)
   ↓
정답 (Target)과 비교
   ↓
얼마나 틀렸는지 숫자 하나로 (Loss)
```

- **Prediction**: 모델이 낸 답
- **Target**: 실제 정답
- **Loss**: 둘이 **얼마나 다른지**를 나타내는 **숫자 하나**. 작을수록 좋음

Day 3에서 랜덤 Weight는 "엉뚱한 출력"을 낸다고 했죠. 그 "엉뚱함"을 숫자로 잰 게 Loss입니다.
Day 7에서는 이 숫자를 **줄이는 방향으로** Weight를 고칩니다. 그래서 Loss는 학습의 **나침반**이에요.

## 2. 숫자를 맞히는 문제: MSE

Day 3(섭씨→화씨), Day 4(sin 곡선) 맛보기에서 썼던 Loss가 이것입니다.

```text
MSE = 평균( (예측 - 정답)² )
```

In [ ]:
pred   = torch.tensor([30.0, 98.0, 215.0])   # 모델이 예측한 화씨
target = torch.tensor([32.0, 98.6, 212.0])   # 실제 화씨

mse_manual = ((pred - target) ** 2).mean()
print("직접 계산 :", round(mse_manual.item(), 3))
print("F.mse_loss:", round(F.mse_loss(pred, target).item(), 3))

MSE는 **"숫자 하나를 맞히는"** 문제(회귀)에 잘 맞습니다.

그런데 LLM이 하는 일은 **"Vocabulary 중에서 하나를 고르는"** 문제(분류)예요.
`"오늘 날씨가" 다음은 [좋다, 춥다, 덥다, 맑다, 자동차] 중 무엇?`
Day 5에서 본 것처럼 모델은 각 후보의 **확률**을 내고, 정답은 그중 **하나의 번호**입니다.

이럴 때 쓰는 Loss가 **Cross Entropy**입니다.

## 3. 핵심 아이디어: `-log(p)`

정답 Token에 모델이 준 확률을 `p`라고 합시다.

```text
Loss = -log(p)
```

이게 Cross Entropy의 전부입니다. 왜 이런 모양일까요?

In [ ]:
p = torch.linspace(0.001, 1.0, 500)

plt.figure(figsize=(7, 4))
plt.plot(p, -torch.log(p), linewidth=2.5)
for pv, label in [(0.9, "확신하고 맞힘"), (0.5, "반반"), (0.05, "확신하고 틀림")]:
    lv = -math.log(pv)
    plt.scatter([pv], [lv], s=60, zorder=3)
    plt.annotate(f"{label}\np={pv} → loss={lv:.2f}", (pv, lv), textcoords="offset points",
                 xytext=(10, 10), fontsize=10)
plt.xlabel("정답 Token에 준 확률 p")
plt.ylabel("Loss = -log(p)")
plt.title("-log(p): 정답에 높은 확률을 줄수록 Loss가 작다", fontsize=12)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
for p_ in [1.0, 0.9, 0.5, 0.1, 0.01, 0.0001]:
    print(f"정답 확률 {p_:>7}  →  loss = {max(0.0, -math.log(p_)):.3f}")

**관찰 포인트**
- `p = 1` (완벽하게 맞힘) → Loss **0**
- `p`가 작아질수록 Loss가 **급격하게** 커짐 → 확신하고 틀리면 **엄청난 벌점**
- 절대 음수가 되지 않음

> 💡 MSE로 확률을 비교하면 `0.01`과 `0.0001`의 차이가 거의 없어 보이지만,
> `-log`는 그 차이를 `4.6 → 9.2`로 크게 벌려서 "정말 많이 틀렸다"고 강하게 알려줍니다.
> 이 강한 신호 덕분에 분류 문제에서 학습이 훨씬 잘 됩니다.

## 4. Cross Entropy 손으로 계산하기

모델은 확률이 아니라 **점수(logits)** 를 냅니다. 그래서 순서는:

```text
logits  →  softmax  →  정답 위치의 확률 꺼내기  →  -log
```

In [ ]:
vocab  = ["좋다", "춥다", "덥다", "맑다", "자동차"]
logits = torch.tensor([3.0, 2.0, 1.0, 0.5, -2.0])
target = 0    # 정답: "좋다"

probs = F.softmax(logits, dim=-1)
p_correct = probs[target]
loss_manual = -torch.log(p_correct)

print("1) 확률         :", r(probs))
print("2) 정답 확률     :", round(p_correct.item(), 4), f'("{vocab[target]}")')
print("3) -log(정답 확률):", round(loss_manual.item(), 4))

### 4-1. PyTorch의 `F.cross_entropy`

`F.cross_entropy(logits, target)`이 위 3단계를 한 번에 합니다.
입력 모양이 정해져 있어요.
- `logits`: `(N, V)` → N개 샘플, 각각 V개 후보의 **점수**
- `target`: `(N,)` → 각 샘플의 정답 **번호** (정수)

In [ ]:
loss_torch = F.cross_entropy(logits.unsqueeze(0), torch.tensor([target]))   # (1, 5), (1,)
print("직접 계산      :", round(loss_manual.item(), 4))
print("F.cross_entropy:", round(loss_torch.item(), 4))

### 4-2. ⚠️ 가장 흔한 실수: 확률을 넣으면 안 된다

`F.cross_entropy`는 **logits를 받아서 내부에서 softmax를 직접** 합니다.
이미 softmax한 확률을 넣으면 softmax가 **두 번** 적용돼서 틀린 값이 나와요. 에러도 안 납니다!

In [ ]:
right = F.cross_entropy(logits.unsqueeze(0), torch.tensor([0]))
wrong = F.cross_entropy(probs.unsqueeze(0), torch.tensor([0]))     # 확률을 넣음 (실수)

print("logits 넣음 (올바름):", round(right.item(), 4))
print("probs  넣음 (실수)  :", round(wrong.item(), 4), " ← 에러 없이 엉뚱한 값")

### 4-3. 같은 logits, 정답이 다르면?

In [ ]:
for t, w in enumerate(vocab):
    l = F.cross_entropy(logits.unsqueeze(0), torch.tensor([t]))
    print(f"정답이 '{w}'라면  →  정답 확률 {probs[t].item()*100:5.1f}%  →  loss {l.item():.3f}")

모델이 "좋다"에 63%를 줬는데 정답이 "자동차"라면 Loss가 5.5나 됩니다.
**모델이 정답에 준 확률만** Loss를 결정한다는 걸 기억하세요.

## 5. 여러 샘플: 평균 Loss

샘플이 여러 개면 각각 Loss를 구한 뒤 **평균**을 냅니다. (`F.cross_entropy`의 기본 동작)

In [ ]:
logits_batch = torch.tensor([
    [3.0, 2.0, 1.0, 0.5, -2.0],   # 샘플 1
    [0.1, 0.2, 4.0, 0.0,  0.0],   # 샘플 2
    [1.0, 1.0, 1.0, 1.0,  1.0],   # 샘플 3 (모든 점수가 같음)
])
targets = torch.tensor([0, 2, 4])  # 정답: 좋다, 덥다, 자동차

each = F.cross_entropy(logits_batch, targets, reduction="none")   # 평균 내지 않고 각각
print("샘플별 loss:", r(each))
print("평균       :", round(each.mean().item(), 4))
print("기본 동작   :", round(F.cross_entropy(logits_batch, targets).item(), 4))

## 6. 기준점: 아무것도 모르는 모델의 Loss = `log(V)`

샘플 3을 보세요. 점수가 모두 같으면 Softmax 결과는 **모든 후보에 1/V씩** 똑같이 나눠줍니다.
그러면 정답 확률은 `1/V`이고, Loss는

```text
-log(1/V) = log(V)
```

**학습을 하나도 안 한 모델의 Loss는 대략 `log(Vocabulary 크기)`** 입니다.

In [ ]:
for V in [5, 65, 50257]:
    print(f"V = {V:>6}  →  log(V) = {math.log(V):.3f}")

- `V = 65`: 4주차 mini GPT(셰익스피어 문자 단위)의 Vocabulary 크기 → **학습 시작 Loss ≈ 4.17**
- `V = 50257`: GPT-2의 Vocabulary 크기 → 시작 Loss ≈ 10.8

> 💡 4주차에 학습을 돌렸을 때 첫 Loss가 4.17 근처면 **모델이 정상적으로 초기화됐다**는 뜻이에요.
> 이보다 훨씬 크면 초기화가 잘못된 것이고, 학습이 진행되며 이 숫자보다 **작아져야** 뭔가 배우고 있는 거예요.

In [ ]:
# 진짜로 확인: 랜덤 Linear Layer로 vocab 65짜리 예측을 해보면?
V = 65
lm_head = torch.nn.Linear(32, V)
x = torch.randn(1000, 32) * 0.1          # 작은 입력 → logits가 거의 비슷
y = torch.randint(0, V, (1000,))         # 랜덤 정답
print("랜덤 모델 loss:", round(F.cross_entropy(lm_head(x), y).item(), 3), "   log(65) =", round(math.log(65), 3))

## 7. LLM과의 연결: `(B, T, V)` 출력으로 Loss 계산하기

### 7-1. 정답(Target)은 "한 칸 밀린 문장"

GPT는 **각 위치에서 다음 Token**을 맞힙니다. 그래서 정답은 입력을 한 칸 민 것이에요.

```text
문장:  나는  오늘  학교  에

x  :  나는  오늘  학교
y  :  오늘  학교  에
       ↑     ↑     ↑
   "나는" 다음 / "나는 오늘" 다음 / "나는 오늘 학교" 다음
```

(커리큘럼 5-3에 `y: 는 오늘 학교 에`라고 적혀 있는데, 위처럼 `y: 오늘 학교 에`가 맞아요.)

In [ ]:
words = ["나는", "오늘", "학교", "에"]
stoi = {w: i for i, w in enumerate(words)}
ids = torch.tensor([stoi[w] for w in words])

x = ids[:-1]     # 마지막 빼고
y = ids[1:]      # 처음 빼고
print("x:", x.tolist(), [words[i] for i in x])
print("y:", y.tolist(), [words[i] for i in y])

### 7-2. shape 맞추기: `(B, T, V)` → `(B*T, V)`

GPT 출력은 `(B, T, V)`, 정답은 `(B, T)`입니다.
그런데 `F.cross_entropy`는 `(N, V)`와 `(N,)`을 원하죠.

**해결**: B와 T를 하나로 합쳐서 "B×T개의 샘플"로 봅니다. 위치 하나하나가 독립적인 분류 문제니까요.

In [ ]:
B, T, V = 2, 3, 5
logits = torch.randn(B, T, V)           # 모델 출력
targets = torch.randint(0, V, (B, T))   # 정답 Token 번호

print("logits :", tuple(logits.shape), " targets:", tuple(targets.shape))

try:
    F.cross_entropy(logits, targets)
except Exception as e:
    print("그대로 넣으면 →", type(e).__name__)

logits_flat  = logits.view(B * T, V)
targets_flat = targets.view(B * T)
loss = F.cross_entropy(logits_flat, targets_flat)

print("펼친 뒤 :", tuple(logits_flat.shape), tuple(targets_flat.shape))
print("loss    :", round(loss.item(), 4))

> 💡 `(B, T, V)`를 그대로 넣으면 PyTorch는 **두 번째 축(T)을 클래스 축**으로 착각합니다.
> 그래서 에러가 나거나, 우연히 shape가 맞으면 **에러 없이 틀린 값**이 나와요.

4주차에 볼 nanoGPT의 Loss 계산 코드가 정확히 이 모양입니다.

```python
B, T, C = logits.shape
logits = logits.view(B * T, C)
targets = targets.view(B * T)
loss = F.cross_entropy(logits, targets)
```

## 8. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** 모델이 정답 Token에 **확률 0.25**를 줬습니다. Cross Entropy Loss는?
`math.log`를 써서 계산식으로 적어도 되고, 숫자로 적어도 됩니다. (소수 셋째 자리까지 맞으면 정답)

In [ ]:
my_answer = "?"   # TODO

assert isinstance(my_answer, (int, float)), "숫자로 적어주세요"
assert abs(my_answer - (-math.log(0.25))) < 1e-3, "Loss = -log(정답 확률) 을 떠올려보세요"
print("Q1 정답!")

**Q2.** `F.cross_entropy`를 쓰지 않고 Cross Entropy를 직접 구현하세요.
- `logits`: `(N, V)`, `targets`: `(N,)` → 평균 Loss (scalar) 반환
- 힌트: `F.softmax(..., dim=-1)` 또는 `F.log_softmax(..., dim=-1)`, 그리고 `probs[torch.arange(N), targets]`로 각 행의 정답 위치 값 꺼내기

In [ ]:
def my_cross_entropy(logits, targets):
    return None  # TODO

logits = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))
assert torch.allclose(my_cross_entropy(logits, targets), F.cross_entropy(logits, targets), atol=1e-5)
print("Q2 정답!")

**Q3.** Vocabulary가 **1000개**인 모델을 막 만들었습니다(학습 전, 모든 후보에 거의 같은 확률).
첫 Loss는 대략 얼마일까요? 소수 첫째 자리까지 맞으면 정답입니다.

In [ ]:
my_guess = "?"   # TODO: 숫자

assert isinstance(my_guess, (int, float)), "숫자로 적어주세요"
assert abs(my_guess - math.log(1000)) < 0.05, "6절의 기준점을 떠올려보세요"
print(f"Q3 정답! log(1000) = {math.log(1000):.3f}")

**Q4.** GPT 출력 `logits (B=4, T=8, V=65)`와 정답 `targets (B=4, T=8)`로 Loss를 계산하세요.
7-2를 참고해서 shape를 맞춰야 합니다.

In [ ]:
B, T, V = 4, 8, 65
logits = torch.randn(B, T, V)
targets = torch.randint(0, V, (B, T))

loss = None  # TODO

expected = F.cross_entropy(logits.reshape(-1, V), targets.reshape(-1))
assert loss is not None and loss.shape == torch.Size([]), "Loss는 숫자 하나(scalar)여야 해요"
assert torch.allclose(loss, expected), "shape를 어떻게 맞췄는지 다시 확인해보세요"
print("Q4 정답! loss =", round(loss.item(), 3))

## 9. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. Loss는 무엇이고, 왜 숫자 하나로 나와야 하는가?
2. Cross Entropy를 한 줄로 설명하면? (힌트: 정답 Token의 확률)
3. 모델이 확신하고 틀리면 Loss는 어떻게 되는가? 왜 그게 좋은가?
4. `F.cross_entropy`에 softmax한 확률을 넣으면 왜 안 되는가?
5. Vocabulary가 65개인 mini GPT의 첫 Loss가 약 4.17이 나오는 이유는?
6. GPT 학습에서 정답 `y`는 입력 `x`와 어떤 관계인가?

**내 답:**

1.
2.
3.
4.
5.
6.